# Chronos-Bolt embedding probe

Residual forecasting was blind to these breaks (they're variance/AR-structure changes, not level shifts — see `chronos.ipynb`). This probe uses Chronos as a **feature extractor** instead: pull the encoder's contextualized summary of `x_hist` and of `x_online`, and let a small classifier learn the break from the *shift* between them.

**Per series**
```
emb_hist   = embed(x_hist)[:, -1, :]     # last patch = whole-window summary (512-d)
emb_online = embed(x_online)[:, -1, :]
feat       = concat(emb_hist, emb_online, emb_online - emb_hist)   # 1536-d
```

**Eval.** L2-logistic regression, 5-fold out-of-fold predictions (honest, no leakage). series-level ROC-AUC + TS-AUC after broadcasting the OOF prob across online steps. Compare vs value detector ~0.55 / bank 0.6064.

Bidirectional T5 encoder + Bolt right-aligns real data, so the last patch attends over the whole window and avoids left-pad contamination.

## Config

In [2]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

from tools import emit_train_datasets, local_ts_auc, PATH

MODEL    = "amazon/chronos-bolt-small"
CTX      = 512            # cap window length fed to the encoder (both hist & online)
N_SERIES = 2000          # more series -> the classifier has something to learn from
BATCH    = 64
DEVICE   = ("mps" if torch.backends.mps.is_available()
            else "cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)

device mps


## Load data (subset)

In [3]:
y_train_index = pd.read_parquet(f"{PATH}y_train_index.parquet")
ids = list(y_train_index.index)[:N_SERIES]
id_set = set(ids)

X = pd.read_parquet(f"{PATH}X_train.parquet")
X = X[X.index.get_level_values("id").isin(id_set)]
print("series", len(ids), "| break_rate", (y_train_index.loc[ids, "tau_index"] != -1).mean())

series 2000 | break_rate 0.515


## Load Chronos-Bolt (once)

In [4]:
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    MODEL, device_map=DEVICE, torch_dtype=torch.float32,
)
print("loaded", MODEL)

/Users/minqi/miniconda3/envs/structural-break/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 143/143 [00:00<00:00, 28139.13it/s]

loaded amazon/chronos-bolt-small


## Embedding extractor

In [5]:
@torch.no_grad()
def embed_batch(arrs):
    """arrs: list of 1-D float arrays -> [len(arrs), 512] last-patch embeddings."""
    ctx = [torch.tensor(a[-CTX:], dtype=torch.float32) for a in arrs]
    emb, _ = pipeline.embed(ctx)      # [B, patches, 512]
    return emb[:, -1, :].float().cpu().numpy()   # last patch = whole-window summary

def embed_all(arrs):
    out = []
    for i in range(0, len(arrs), BATCH):
        out.append(embed_batch(arrs[i:i + BATCH]))
        print(f"  {min(i + BATCH, len(arrs))}/{len(arrs)}", end="\r")
    print()
    return np.vstack(out)

## Build embedding features

In [6]:
hist_arrs, online_arrs, labels, order_ids = [], [], [], []
for sid, x_hist, x_online, tau in emit_train_datasets(X, y_train_index, selected_ids=ids):
    hist_arrs.append(x_hist)
    online_arrs.append(x_online)
    labels.append(int(tau is not None))
    order_ids.append(sid)

print("embedding hist...");   E_hist   = embed_all(hist_arrs)
print("embedding online..."); E_online = embed_all(online_arrs)

Xemb = np.hstack([E_hist, E_online, E_online - E_hist])   # [n, 1536]
y    = np.array(labels)
print("Xemb", Xemb.shape, "| break_rate", y.mean())

embedding hist...
  2000/2000
embedding online...
  2000/2000
Xemb (2000, 1536) | break_rate 0.515


## Eval — 5-fold OOF logistic regression

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

oof = np.zeros(len(y))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for tr, va in skf.split(Xemb, y):
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(C=0.1, max_iter=2000))
    clf.fit(Xemb[tr], y[tr])
    oof[va] = clf.predict_proba(Xemb[va])[:, 1]

print(f"series-level OOF ROC-AUC: {roc_auc_score(y, oof):.4f}   "
      f"(value detector ~0.55, bank 0.6064)")

series-level OOF ROC-AUC: 0.5135   (value detector ~0.55, bank 0.6064)


## Eval — TS-AUC (OOF prob broadcast across online steps)

In [8]:
prob = dict(zip(order_ids, oof))
rows_id, rows_time, rows_pred = [], [], []
for sid in order_ids:
    sub = X.loc[sid]
    on_times = sub.index[sub["period"].to_numpy() == 2]
    rows_id.extend([sid] * len(on_times))
    rows_time.extend(list(on_times))
    rows_pred.extend([prob[sid]] * len(on_times))

pred = pd.DataFrame({"prediction": rows_pred}, index=pd.MultiIndex.from_arrays(
    [rows_id, rows_time], names=["id", "time"])).sort_index()
ytar = pd.read_parquet(f"{PATH}y_train.parquet")
ytar = ytar.loc[ytar.index.get_level_values("id").isin(id_set)]
print(f"TS-AUC (broadcast): {local_ts_auc(ytar, pred):.4f}   vs bank baseline 0.6064")

TS-AUC (broadcast): 0.4931   vs bank baseline 0.6064
